## Task 2

In [2]:
from __future__ import annotations


import argparse
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from z3 import And, Bool, If, Implies, Not, Or, PbEq, PbLe, Solver, sat


@dataclass
class SortingSolution:
    horizon: int
    moves: List[Tuple[int, str, str, str]]



In [3]:
def solve_sorting(
    bricks: List[str],
    positions: List[str],
    initial: Dict[str, str],
    goal: Dict[str, str],
    horizon: int,
) -> Optional[SortingSolution]:

    if set(initial) != set(bricks) or set(goal) != set(bricks):
        raise ValueError("initial and goal must define every brick")
    if any(initial[b] not in positions or goal[b] not in positions for b in bricks):
        raise ValueError("all initial and goal positions must be valid")
    if len(set(initial.values())) != len(initial):
        raise ValueError("initial positions must be distinct")
    if len(set(goal.values())) != len(goal):
        raise ValueError("goal positions must be distinct")

    solver = Solver()
    at = {
        (b, p, t): Bool(f"at_{b}_{p}_{t}")
        for b in bricks
        for p in positions
        for t in range(horizon + 1)
    }
    move = {
        (b, p, q, t): Bool(f"move_{b}_{p}_{q}_{t}")
        for b in bricks
        for p in positions
        for q in positions
        if p != q
        for t in range(horizon)
    }
    moved = {
        (b, t): Bool(f"moved_{b}_{t}")
        for b in bricks
        for t in range(horizon)
    }

    # Every brick occupies exactly one position at every time.
    for b in bricks:
        for t in range(horizon + 1):
            solver.add(PbEq([(at[b, p, t], 1) for p in positions], 1))

    # Every position contains at most one brick.
    for p in positions:
        for t in range(horizon + 1):
            solver.add(PbLe([(at[b, p, t], 1) for b in bricks], 1))

    # Initial and goal states.
    for b in bricks:
        for p in positions:
            solver.add(at[b, p, 0] == (p == initial[b]))
            solver.add(at[b, p, horizon] == (p == goal[b]))

    for b in bricks:
        for t in range(horizon):
            brick_moves = [
                move[b, p, q, t]
                for p in positions
                for q in positions
                if p != q
            ]
            solver.add(moved[b, t] == Or(brick_moves))

            # At most one brick is moved at a time-step.
            solver.add(PbLe([(moved[x, t], 1) for x in bricks], 1))

            # A move must start from the brick's current position.
            for p in positions:
                for q in positions:
                    if p != q:
                        solver.add(Implies(move[b, p, q, t], at[b, p, t]))

            # State transition: stay, or arrive from exactly one other place.
            for q in positions:
                incoming = [move[b, p, q, t] for p in positions if p != q]
                stays = And(at[b, q, t], Not(moved[b, t]))
                solver.add(at[b, q, t + 1] == Or([stays] + incoming))

    if solver.check() != sat:
        return None

    model = solver.model()
    result: List[Tuple[int, str, str, str]] = []
    for t in range(horizon):
        for b in bricks:
            for p in positions:
                for q in positions:
                    if p != q and model.evaluate(move[b, p, q, t], model_completion=True):
                        result.append((t, b, p, q))
    return SortingSolution(horizon=horizon, moves=result)




In [4]:
def find_minimum_solution(
    bricks: List[str],
    positions: List[str],
    initial: Dict[str, str],
    goal: Dict[str, str],
    max_horizon: int,
) -> Optional[SortingSolution]:
    """Try horizons 0..max_horizon and return the first feasible solution."""
    for horizon in range(max_horizon + 1):
        solution = solve_sorting(bricks, positions, initial, goal, horizon)
        if solution is not None:
            return solution
    return None




In [7]:
def main():
    parser = argparse.ArgumentParser(
        description="Solve Task II sorting model"
    )

    parser.add_argument(
        "--max-horizon",
        type=int,
        default=10
    )

    args = parser.parse_args(args=[])

    bricks = ["b1", "b2", "b3"]
    positions = ["p1", "p2", "p3", "p4"]

    initial = {
        "b1": "p1",
        "b2": "p2",
        "b3": "p3"
    }

    goal = {
        "b1": "p3",
        "b2": "p1",
        "b3": "p2"
    }

    solution = find_minimum_solution(
        bricks,
        positions,
        initial,
        goal,
        args.max_horizon
    )

    if solution is None:
        print("No solution within the requested horizon.")
        return

    print("Minimum number of moves:", len(solution.moves))

    for t, brick, source, target in solution.moves:
        print(f"t={t}: {brick}: {source} -> {target}")

In [8]:
if __name__ == "__main__":
    main()

Minimum number of moves: 4
t=0: b1: p1 -> p4
t=1: b2: p2 -> p1
t=2: b3: p3 -> p2
t=3: b1: p4 -> p3
